# Embeddings merging strategies analysis

In [224]:
import pandas as pd
import os
import numpy as np
from typing import Literal


In [5]:
base_path = "../"

#### Load data

In [157]:
df = pd.read_csv(os.path.join(base_path, "gridsearch_merging_strategy.csv"))
df.head()

,DNABERTBACTSTRAT,NT2PHAGESTRAT,NT2BACTSTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,MEGADNABACTSTRAT,F1Score1,Average
0,TruncateStrategy,TruncateStrategy,TruncateStrategy,TfidfStrategy,Tf4idfStrategy,MaxStrategy,0.910256,0.9102
1,TruncateStrategy,TruncateStrategy,TruncateStrategy,MaxStrategy,Tf4idfStrategy,MaxStrategy,0.920198,0.9201
2,TruncateStrategy,TruncateStrategy,TruncateStrategy,MaxStrategy,TruncateStrategy,TruncateStrategy,0.923039,0.9230
3,TruncateStrategy,TruncateStrategy,TruncateStrategy,TruncateStrategy,MaxStrategy,TruncateStrategy,0.925735,0.9257
4,TruncateStrategy,TruncateStrategy,TruncateStrategy,TfidfStrategy,TruncateStrategy,Tf4idfStrategy,0.915788,0.9157


## Analysis

In [158]:
th_high = 0.95
th_low = 0.80

In [159]:
df.sort_values(axis=0, by="Average", ascending=False, inplace=True, ignore_index=True)
df.head()

,DNABERTBACTSTRAT,NT2PHAGESTRAT,NT2BACTSTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,MEGADNABACTSTRAT,F1Score1,Average
0,MaxStrategy,TruncateStrategy,Tf4idfStrategy,TfidfStrategy,MaxStrategy,MaxStrategy,0.953854,0.9538
1,MaxStrategy,TruncateStrategy,TfidfStrategy,MaxStrategy,Tf4idfStrategy,MaxStrategy,0.953701,0.9537
2,MaxStrategy,TKPertStrategy,TruncateStrategy,TfidfStrategy,TKPertStrategy,TruncateStrategy,0.953565,0.9535
3,MaxStrategy,TfidfStrategy,TfidfStrategy,TKPertStrategy,TruncateStrategy,TruncateStrategy,0.953387,0.9533
4,MaxStrategy,TfidfStrategy,TruncateStrategy,TfidfStrategy,MaxStrategy,MaxStrategy,0.953267,0.9532


In [160]:
df_best = df[df["Average"] >= th_high]
print(f"Number of elements selected as bests: {len(df_best)}")

df_worse = df[df["Average"] < th_low]
print(f"Number of elements selected as worse: {len(df_worse)}")

Number of elements selected as bests: 100
Number of elements selected as worse: 158


In [225]:
DF_COLUMNS = ["NT2PHAGESTRAT", "MEGADNAPHAGESTRAT", "DNABERTPHAGESTRAT", "NT2BACTSTRAT", "MEGADNABACTSTRAT", "DNABERTBACTSTRAT"]
strategies = np.unique(df[DF_COLUMNS].values)

def make_pretty(styler: pd.io.formats.style.Styler, title: str, cmap: str = "RdYlGn", vmin = None, vmax = None, percentage: bool = False, axis: Literal[0] | None = 0):
    styler.set_caption(title)
    if percentage:
        styler.format(lambda x: f"{x*100:.2f}%") # type: ignore
    else:
        styler.format(precision=3)
    styler.background_gradient(axis=axis, cmap=cmap, vmin=vmin, vmax=vmax)
    return styler


### Single column analysis

How many of each of them are in the top and bottom

In [208]:
table_best = pd.DataFrame(columns=DF_COLUMNS, index=strategies, dtype=float).fillna(0)
table_worse = pd.DataFrame(columns=DF_COLUMNS, index=strategies, dtype=float).fillna(0)

for column in DF_COLUMNS:
    table_best[column] = df_best[column].value_counts(normalize=True)
    table_worse[column] = df_worse[column].value_counts(normalize=True)

table_best.fillna(0, inplace=True)
table_worse.fillna(0, inplace=True)

In [235]:
table_best.style.pipe(make_pretty, title=f"Best strategies (f1 >= {th_high})", percentage=True)

,NT2PHAGESTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,NT2BACTSTRAT,MEGADNABACTSTRAT,DNABERTBACTSTRAT
MaxStrategy,20.93%,25.58%,20.54%,13.18%,18.99%,30.62%
TKPertStrategy,25.97%,18.99%,25.97%,7.75%,1.16%,0.00%
Tf4idfStrategy,15.12%,16.28%,19.38%,31.01%,36.82%,8.91%
TfidfStrategy,15.12%,15.50%,17.83%,27.91%,27.52%,55.81%
TruncateStrategy,22.87%,23.64%,16.28%,20.16%,15.50%,4.65%


In [234]:
table_worse.style.pipe(make_pretty, title=f"Worse strategies (f1 < {th_low})", cmap="RdYlGn_r", percentage=True)

,NT2PHAGESTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,NT2BACTSTRAT,MEGADNABACTSTRAT,DNABERTBACTSTRAT
MaxStrategy,25.95%,26.58%,19.62%,1.27%,0.63%,2.53%
TKPertStrategy,18.99%,25.32%,36.08%,8.23%,0.00%,0.00%
Tf4idfStrategy,15.19%,9.49%,13.92%,46.20%,55.70%,13.29%
TfidfStrategy,15.19%,8.23%,16.46%,43.04%,43.04%,84.18%
TruncateStrategy,24.68%,30.38%,13.92%,1.27%,0.63%,0.00%


### Single column analysis: Average score for each strategy

In [212]:
weights_table = pd.DataFrame(columns=DF_COLUMNS, index=strategies, dtype=float).fillna(0)

for column in DF_COLUMNS:
    for strat in strategies:
        weights_table.loc[strat, column] = df[df[column] == strat]["Average"].mean()

weights_table.style.pipe(make_pretty, title="Average score", cmap="RdYlGn")

,NT2PHAGESTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,NT2BACTSTRAT,MEGADNABACTSTRAT,DNABERTBACTSTRAT
MaxStrategy,0.913,0.914,0.914,0.930,0.927,0.925
TKPertStrategy,0.918,0.914,0.916,0.907,0.920,nan
Tf4idfStrategy,0.915,0.917,0.915,0.904,0.898,0.902
TfidfStrategy,0.915,0.918,0.915,0.904,0.899,0.897
TruncateStrategy,0.915,0.913,0.916,0.930,0.932,0.925


In [213]:
weights_table = pd.DataFrame(columns=DF_COLUMNS, index=strategies, dtype=float).fillna(0)
df_top_bottom = df_best = df[(df["Average"] >= th_high) | (df["Average"] < th_low)]

for column in DF_COLUMNS:
    for strat in strategies:
        weights_table.loc[strat, column] = df_top_bottom[df_top_bottom[column] == strat]["Average"].mean()

weights_table.style.pipe(make_pretty, title="Average score (only top + bottom)", cmap="RdYlGn", vmin=0.80, vmax=0.95)

,NT2PHAGESTRAT,MEGADNAPHAGESTRAT,DNABERTPHAGESTRAT,NT2BACTSTRAT,MEGADNABACTSTRAT,DNABERTBACTSTRAT
MaxStrategy,0.829,0.849,0.856,0.940,0.946,0.941
TKPertStrategy,0.879,0.819,0.815,0.848,0.951,nan
Tf4idfStrategy,0.850,0.894,0.880,0.804,0.802,0.805
TfidfStrategy,0.851,0.900,0.860,0.800,0.798,0.803
TruncateStrategy,0.846,0.822,0.865,0.943,0.947,0.950
